# Chapter 10 — Compile the Program

**Book alignment:** DSPy From First Principles, Chapter 10

**Question this notebook isolates:** Does compilation preserve the baseline and return a distinct candidate under a declared boundary, without saying anything about quality?

In [ ]:
import copy
from pathlib import Path
import random
import sys

random.seed(13)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy  # imported and constructed only; compile is never executed against a model

from common.data import canonical_split
from common.dspy_program import EditorialRewriteProgram, dspy_editorial_metric_v1
from common.fingerprints import fingerprint

## Audit before and after

Three snapshots, not two: baseline-before, baseline-after, and candidate state. Constructing (never executing) the declared `BootstrapFewShot` configuration, then attaching recorded-style demonstrations to a copied candidate, must leave the accepted baseline bit-identical.

In [ ]:
split = canonical_split()
baseline = EditorialRewriteProgram()
baseline_before = fingerprint(baseline.dump_state())

# Declared optimizer configuration from the chapter; constructed, never compiled.
optimizer = dspy.BootstrapFewShot(
    metric=dspy_editorial_metric_v1,
    max_bootstrapped_demos=2,
    max_labeled_demos=0,
)

# Candidate as a distinct object carrying recorded-style demonstrations
# sourced from the first two training cases (the book's measured outcome).
candidate = copy.deepcopy(baseline)
demo_sources = list(split.train[:2])
for pred_name, predictor in candidate.named_predictors():
    predictor.demos = [
        dspy.Example(
            sentence=c.sentence, goal=c.goal, context=c.context,
            rewritten_text=c.reference_rewrite,
        ).with_inputs("sentence", "goal", "context")
        for c in demo_sources
    ]

baseline_after = fingerprint(baseline.dump_state())
candidate_fp = fingerprint(candidate.dump_state())

def demo_count(program) -> int:
    return sum(len(predictor.demos or []) for _, predictor in program.named_predictors())

({
    "baseline_before": baseline_before[:12],
    "baseline_after": baseline_after[:12],
    "candidate": candidate_fp[:12],
    "baseline_demos": demo_count(baseline),
    "candidate_demos": demo_count(candidate),
})

In [ ]:
assert baseline_before == baseline_after  # accepted program untouched in passing
assert candidate is not baseline
assert candidate_fp != baseline_before  # state transition actually happened
assert demo_count(baseline) == 0
assert demo_count(candidate) == 6  # two demos x three predictors

print("baseline unchanged; candidate is a separate, changed object")

## Available evidence is not consumed evidence

Twenty-six training cases are admissible to this configuration, but a greedy quota of two fills from the first qualifying examples. Replay that traversal rule deterministically and check the holdout never enters.

In [ ]:
quota = 2
visible = list(split.train_ids)
consumed, untouched = [], []
for case_id in visible:
    (consumed if len(consumed) < quota else untouched).append(case_id)

({
    "admissible": len(visible),
    "consumed": consumed,
    "untouched": len(untouched),
    "untouched_families": sorted({c.source_group for c in split.train if c.case_id in untouched}),
})

In [ ]:
assert consumed == ["ed-001", "ed-002"]
assert len(untouched) == 24
assert set(visible).isdisjoint(split.dev_ids)
assert set(visible).isdisjoint(split.holdout_ids)
assert len(consumed) / len(visible) < 0.08

print(f"{len(consumed)} of {len(visible)} admissible cases shaped the candidate")
print("dev reserved; holdout excluded and uninspected; no promotion")

## What we earned

Compilation happens inside an inspectable boundary: the baseline is unchanged, the candidate is a distinct object with six demonstrations from two cases, dev stays reserved, and holdout stays sealed. This establishes a state transition — nothing about quality.

Notebook 11 / Chapter 11 asks the deferred question: does the candidate actually help?